# AGENTIC AI
## SESSION 4

Assignment questions and answers


## Question 1

AI-AUGMENTED LEARNING

STEP 1 — BUILD WITH AI

Write a Python program implementing a simplified ReAct (Reason + Action)
loop for a food-delivery customer-support agent processing a complaint.

Requirements: - Execute at least two complete Reason → Action →
Observation cycles before the final response, printing each cycle’s
thought, action call, and observation. - Include at least two callable
tools, such as lookup_order_status(order_id) and
check_refund_eligibility(order_id), invoked during the action step. -
Handle a tool error or ‘not found’ result and use that observation in
the next reasoning step instead of crashing.

STEP 2 — TEST & DEBUG (WITHOUT AI)

Test independently, identify at least one bug, limitation, or
improvement in the AI-generated solution, and fix it yourself.

SUBMISSION REQUIREMENTS: - Exact prompt(s) given to the AI tool. - AI’s
original code and corrected version. - A 3–4 line note explaining what
changed and why it was needed.

**Answer:**

### Prompt Given to the AI Tool
"Write a Python program implementing a simplified ReAct loop for a food-delivery customer-support agent processing a complaint. The agent must execute at least two Thought -> Action -> Observation cycles before giving a final response. Include two callable tools: lookup_order_status(order_id) and check_refund_eligibility(order_id). Handle missing/error cases gracefully within the loop instead of crashing."

---

### AI's Original Code

```python
def lookup_order_status(order_id):
    database = {"101": "Delivered", "102": "Delayed"}
    return database.get(order_id, "Order not found")


def check_refund_eligibility(order_id):
    database = {"101": True, "102": False}
    return database.get(order_id, "Error: Unknown order")


def react_agent(order_id, complaint):
    tools = {
        "lookup_order_status": lookup_order_status,
        "check_refund_eligibility": check_refund_eligibility,
    }

    # Cycle 1
    thought_1 = f"Customer has an issue: '{complaint}'. I need to check the order status first."
    action_1 = ("lookup_order_status", order_id)
    observation_1 = tools[action_1[0]](action_1[1])
    print(
        f"Thought 1: {thought_1}\nAction 1: {action_1[0]}({action_1[1]})\nObservation 1: {observation_1}\n"
    )

    # Cycle 2
    if "not found" in observation_1:
        thought_2 = (
            "Order was not found. I cannot issue a refund without a valid order."
        )
        action_2 = ("check_refund_eligibility", order_id)
        observation_2 = tools[action_2[0]](action_2[1])
    else:
        thought_2 = (
            "Order exists. Now checking if the customer is eligible for a refund."
        )
        action_2 = ("check_refund_eligibility", order_id)
        observation_2 = tools[action_2[0]](action_2[1])

    print(
        f"Thought 2: {thought_2}\nAction 2: {action_2[0]}({action_2[1]})\nObservation 2: {observation_2}\n"
    )

    # Final Answer
    final_answer = (
        f"Unable to process complaint for order {order_id}: {observation_1}"
        if "not found" in observation_1
        else f"Order {order_id} handled. Refund eligible: {observation_2}"
    )
    print(f"Final Answer: {final_answer}")


react_agent("999", "Food never arrived")
```

---

### Corrected Version

```python
def lookup_order_status(order_id):
    database = {"101": "Delivered", "102": "Delayed"}
    if order_id not in database:
        return "Error: Order not found"
    return database[order_id]


def check_refund_eligibility(order_id):
    database = {"101": True, "102": False}
    if order_id not in database:
        return "Error: Order not found"
    return f"Eligible: {database[order_id]}"


def search_customer_orders(customer_name):
    customer_db = {"Alice": "101"}
    return customer_db.get(customer_name, "Error: Customer not found")


def react_agent(user_input):
    tools = {
        "lookup_order_status": lookup_order_status,
        "check_refund_eligibility": check_refund_eligibility,
        "search_customer_orders": search_customer_orders,
    }

    # Cycle 1: Try checking order status
    thought_1 = f"Received complaint for order {user_input['order_id']}. Checking status."
    action_1 = ("lookup_order_status", user_input["order_id"])
    obs_1 = tools[action_1[0]](action_1[1])
    print(
        f"Thought 1: {thought_1}\nAction 1: {action_1[0]}('{action_1[1]}')\nObservation 1: {obs_1}\n"
    )

    # Cycle 2: Handle missing order by fallback search, or check refund
    if "Error" in obs_1:
        thought_2 = "Order ID failed. Attempting lookup via customer name."
        action_2 = ("search_customer_orders", user_input["customer_name"])
        obs_2 = tools[action_2[0]](action_2[1])
    else:
        thought_2 = (
            "Order found. Checking refund eligibility for the complaint."
        )
        action_2 = ("check_refund_eligibility", user_input["order_id"])
        obs_2 = tools[action_2[0]](action_2[1])
    print(
        f"Thought 2: {thought_2}\nAction 2: {action_2[0]}('{action_2[1]}')\nObservation 2: {obs_2}\n"
    )

    # Cycle 3: Process refund if recovered, or finalize
    if "Error" not in obs_2 and action_2[0] == "search_customer_orders":
        recovered_id = obs_2
        thought_3 = f"Found order ID {recovered_id}. Now checking refund eligibility."
        action_3 = ("check_refund_eligibility", recovered_id)
        obs_3 = tools[action_3[0]](action_3[1])
        print(
            f"Thought 3: {thought_3}\nAction 3: {action_3[0]}('{action_3[1]}')\nObservation 3: {obs_3}\n"
        )
        final_response = f"Identified order {recovered_id}. Resolution: {obs_3}"
    elif "Error" in obs_2:
        final_response = "Unable to locate any order records. Please contact live support."
    else:
        final_response = (
            f"Order {user_input['order_id']} status: {obs_1}. Result: {obs_2}"
        )

    print(f"Final Answer: {final_response}")


react_agent(
    {"order_id": "999", "customer_name": "Alice", "issue": "Missing item"}
)
```

---

### Explanation of Changes
1. **Flawed Logic Fixed:** The original code called `check_refund_eligibility` even after learning the order was nonexistent, wasting an action on a guaranteed failure.
2. **Added Fallback Tool:** Introduced `search_customer_orders` so an "Order not found" error triggers an alternative recovery action in Cycle 2.
3. **Dynamic Multi-Cycle Flow:** Ensured the agent adapts its reasoning to errors rationally rather than executing a hardcoded sequence.